# BoTTLE on Colab: profile on a free T4, predict edge cost per task

Runtime → Change runtime type → **T4 GPU**. Run all cells (~10 min).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,power.limit --format=csv

## 1. Install
If the repo is not public yet, upload the `BoTTLE` folder (zip) to the Colab session, unzip it, and `pip install -e BoTTLE[energy]` instead.

In [ ]:
!pip install -q "bottle-bench[energy] @ git+https://github.com/PKM100/BoTTLE.git"
!bottle --version

## 2. Profile every model on the T4 (reference device)
Shape-faithful synthetic stacks, fp16. Large models are profiled on a subset of layers and extrapolated so they fit in 16 GB.

In [ ]:
import subprocess
MODELS = {"smolvla": None, "gr00t-n1.6": None, "gr00t-n1.7": None, "pi0": None, "pi05": None,
          "openvla-oft": 8, "openvla": 8}
for m, max_layers in MODELS.items():
    cmd = ["bottle", "profile", "--model", m, "--hw-name", "t4", "--device", "cuda", "--precision", "fp16",
           "--out", f"profiles/{m}_t4_fp16.json"]
    if max_layers: cmd += ["--max-layers", str(max_layers)]
    print(subprocess.run(cmd, capture_output=True, text=True).stdout)

## 3. Uncalibrated vs calibrated estimate for Jetson Thor

In [ ]:
!bottle estimate --model pi0 --hw t4 jetson-agx-thor
!bottle estimate --model pi0 --hw t4 jetson-agx-thor --profile-dir profiles --calib-mode stage
!bottle estimate --model pi0 --hw t4 jetson-agx-thor --profile-dir profiles --calib-mode overhead

## 4. Cost per successful task across models × hardware
Success rates are published LIBERO averages (OpenVLA, OpenVLA-OFT, pi0, pi0.5, SmolVLA, GR00T N1.6). Replace them with your own sim results if you have them.

In [ ]:
SUCCESS = "smolvla=0.873,pi0=0.942,pi05=0.969,openvla=0.765,openvla-oft=0.971,gr00t-n1.6=0.970,gr00t-n1.7=0.970"  # LIBERO avg
!bottle sweep --hw t4 l4 jetson-agx-orin-64gb jetson-agx-thor jetson-orin-nano-super \
    --precisions fp16 w4a16 --success {SUCCESS} --steps 300 --control-hz 20 \
    --robot-usd-per-hr 10 --profile-dir profiles --out results/sweep_calibrated.csv

In [ ]:
import pandas as pd
df = pd.read_csv("results/sweep_calibrated.csv")
df[["model","hardware","precision","latency_ms","realtime_ok","fits","usd_per_1k_successes","pareto"]].head(20)

## 5. Download

In [ ]:
!zip -qr bottle_results.zip profiles results
from google.colab import files; files.download('bottle_results.zip')